In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GIT_TOKEN_GENAI2 = user_secrets.get_secret("GIT_TOKEN_GENAI2")

In [ ]:
import os
os.environ["GIT_TOKEN_GENAI2"] = GIT_TOKEN_GENAI2

In [ ]:
!git clone https://$GIT_TOKEN_GENAI2@github.com/standing-on-giants/CatVTON-GenAI-Virtual-Try-On.git

In [ ]:
%cd /kaggle/working/CatVTON-GenAI-Virtual-Try-On
!pwd

In [ ]:
!pip install -q \
  git+https://github.com/huggingface/diffusers.git \
  accelerate==0.34.0 \
  transformers==4.50.0 \
  peft==0.17.0 \
  huggingface_hub==0.34.0 \
  omegaconf==2.3.0 \
  fvcore==0.1.5.post20221221 \
  cloudpickle==3.0.0 \
  pycocotools==2.0.8 \
  av==12.3.0

In [ ]:
import os

# Just symlink the missing folder name (agnostic-mask -> agnostic-v3.2)
# We need a writable location for the symlink, so create a thin wrapper dir
os.makedirs("/kaggle/working/VITON-HD/test", exist_ok=True)
os.makedirs("/kaggle/working/VITON-HD/train", exist_ok=True)

DATA = "/kaggle/input/datasets/marquis03/high-resolution-viton-zalando-dataset"

# Symlink all existing folders/files into working (no copying!)
for item in os.listdir(f"{DATA}/test"):
    src = f"{DATA}/test/{item}"
    dst = f"/kaggle/working/VITON-HD/test/{item}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

for item in os.listdir(f"{DATA}/train"):
    src = f"{DATA}/train/{item}"
    dst = f"/kaggle/working/VITON-HD/train/{item}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

# Add the renamed mask folder as a symlink with the correct name
# os.symlink(f"{DATA}/test/agnostic-v3.2", "/kaggle/working/VITON-HD/test/agnostic-mask")

# Download and place the tagged JSONs (these are tiny)
os.system("wget -q https://raw.githubusercontent.com/yisol/IDM-VTON/main/vitonhd_test_tagged.json -O /kaggle/working/VITON-HD/test/vitonhd_test_tagged.json")
os.system("wget -q https://raw.githubusercontent.com/yisol/IDM-VTON/main/vitonhd_train_tagged.json -O /kaggle/working/VITON-HD/train/vitonhd_train_tagged.json")

# Copy the pairs txt files
import shutil
shutil.copy(f"{DATA}/test_pairs.txt", "/kaggle/working/VITON-HD/test_pairs_unpaired.txt")
shutil.copy(f"{DATA}/train_pairs.txt", "/kaggle/working/VITON-HD/train_pairs_unpaired.txt")

print("Done! No data copied, only symlinked.")

In [ ]:
#image name jpg to png fixing

with open("/kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py", "r") as f:
    content = f.read()

content = content.replace(
    "'mask': os.path.join(self.args.data_root_path, 'agnostic-mask', person_img),",
    "'mask': os.path.join(self.args.data_root_path, 'agnostic-mask', person_img.replace('.jpg', '_mask.png')),"
)

with open("/kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py", "w") as f:
    f.write(content)

print("Done! Verify:")
!grep -n "agnostic-mask" /kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py

#Dimension fixing


with open("/kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py", "r") as f:
    content = f.read()

content = content.replace(
    "    mask_np = np.array(mask) / 255\n    repaint_result = person_np * (1 - mask_np) + result_np * mask_np",
    "    mask_np = np.array(mask) / 255\n    if mask_np.ndim == 2:\n        mask_np = mask_np[:, :, np.newaxis]\n    repaint_result = person_np * (1 - mask_np) + result_np * mask_np"
)

with open("/kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py", "w") as f:
    f.write(content)

print("Fixed! Verify:")
!grep -A5 "mask_np = np.array" /kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py

In [ ]:
#Creating binary mask

import os
import numpy as np
from PIL import Image
from tqdm import tqdm

parse_dir = "/kaggle/input/datasets/marquis03/high-resolution-viton-zalando-dataset/test/image-parse-v3"
out_dir = "/kaggle/working/VITON-HD/test/agnostic-mask-binary"
os.makedirs(out_dir, exist_ok=True)

GARMENT_LABELS = [126, 55, 140, 178]  # garment + neck + left arm + right arm

for fname in tqdm(os.listdir(parse_dir)):
    parse = Image.open(os.path.join(parse_dir, fname)).convert('L')
    parse_np = np.array(parse)
    mask = np.zeros_like(parse_np, dtype=np.uint8)
    for label in GARMENT_LABELS:
        mask[parse_np == label] = 255
    out_name = fname.replace('.png', '_mask.png')
    Image.fromarray(mask).save(os.path.join(out_dir, out_name))

print("Done!")

In [ ]:
# Fix symlink
old = "/kaggle/working/VITON-HD/test/agnostic-mask"
if os.path.islink(old) or os.path.exists(old):
    os.remove(old)
os.symlink(out_dir, old)

# Fix filename mismatch in inference.py
# The code does: person_img.replace('.jpg', '_mask.png')
# But our files are named: 00006_00.png (no _mask suffix)
# Rename all files to match expected pattern:
for fname in tqdm(os.listdir(out_dir)):
    if fname.endswith('.png') and '_mask' not in fname:
        old_path = os.path.join(out_dir, fname)
        new_path = os.path.join(out_dir, fname.replace('.png', '_mask.png'))
        os.rename(old_path, new_path)

print("Sample files:", sorted(os.listdir(out_dir))[:3])
# Should show: ['00006_00_mask.png', '00008_00_mask.png', ...]

In [ ]:
#Visualising new binary mask
import os, random
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
person = Image.open("/kaggle/working/VITON-HD/test/image/02532_00.jpg")
new_mask = Image.open(f"{out_dir}/02532_00_mask.png")
axes[0].imshow(person); axes[0].set_title("Person")
axes[1].imshow(new_mask, cmap='gray'); axes[1].set_title("New mask (with arms)")
axes[2].imshow(person); axes[2].imshow(new_mask, cmap='Reds', alpha=0.5); axes[2].set_title("Overlay")
plt.show()

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python inference.py \
  --dataset_name vitonhd \
  --data_root_path /kaggle/working/VITON-HD \
  --output_dir /kaggle/working/output_v4 \
  --dataloader_num_workers 2 \
  --batch_size 4 \
  --seed 555 \
  --mixed_precision fp16 \
  --allow_tf32 \
  --repaint

In [ ]:
import os, random
from PIL import Image
import matplotlib.pyplot as plt

image_dir = "/kaggle/working/VITON-HD/test/image"
cloth_dir = "/kaggle/working/VITON-HD/test/cloth"
result_dir = "/kaggle/working/output_v4/vitonhd-512/unpaired"

with open("/kaggle/working/VITON-HD/test_pairs_unpaired.txt") as f:
    all_pairs = [line.strip().split() for line in f.readlines()]

# Only keep pairs where result exists
available = [
    (im, cl) for im, cl in all_pairs
    if os.path.exists(f"{result_dir}/{im}")
]

sample = random.sample(available, min(10, len(available)))


fig, axes = plt.subplots(len(sample), 3, figsize=(9, len(sample)*3))
for i, (im_name, cl_name) in enumerate(sample):
    axes[i,0].imshow(Image.open(f"{image_dir}/{im_name}"))
    axes[i,0].set_title("Person", fontsize=8)
    axes[i,0].axis("off")

    axes[i,1].imshow(Image.open(f"{cloth_dir}/{cl_name}"))
    axes[i,1].set_title("Cloth", fontsize=8)
    axes[i,1].axis("off")

    axes[i,2].imshow(Image.open(f"{result_dir}/{im_name}"))
    axes[i,2].set_title("Result", fontsize=8)
    axes[i,2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/tryon_comparison.png", dpi=150)
plt.show()

In [ ]:
!pip install -q clean-fid

In [ ]:
!python /kaggle/working/CatVTON-GenAI-Virtual-Try-On/eval.py \
  --gt_folder /kaggle/working/VITON-HD/test/image \
  --pred_folder /kaggle/working/output_v4/vitonhd-512/unpaired \
  --batch_size 16 \
  --num_workers 2

#Results are bad bcoz just 8 images are present in inference output folder. Evaluation is really quick. Inference takes a lot of time

In [ ]:
#To see random samples of person & cloth images together to decide which ones to keep


import os, random
from PIL import Image
import matplotlib.pyplot as plt

image_dir = "/kaggle/working/VITON-HD/test/image"
cloth_dir = "/kaggle/working/VITON-HD/test/cloth"

with open("/kaggle/working/VITON-HD/test_pairs_unpaired.txt") as f:
    all_pairs = [line.strip().split() for line in f.readlines()]

# Sample with their original indices preserved
sample_indices = random.sample(range(len(all_pairs)), 20)
sample = [(idx, all_pairs[idx][0], all_pairs[idx][1]) for idx in sample_indices]

fig, axes = plt.subplots(len(sample), 2, figsize=(6, len(sample)*3))
for i, (idx, im_name, c_name) in enumerate(sample):
    axes[i,0].imshow(Image.open(f"{image_dir}/{im_name}"))
    axes[i,0].set_title(f"[{idx}] {im_name}", fontsize=6)
    axes[i,0].axis("off")
    axes[i,1].imshow(Image.open(f"{cloth_dir}/{c_name}"))
    axes[i,1].set_title(f"[{idx}] {c_name}", fontsize=6)
    axes[i,1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/inspection_grid.png", dpi=150)
plt.show()

# Print a clean index→pair mapping for easy reference
print("\nIndex → (person, cloth)")
print("-" * 50)
for idx, im_name, c_name in sample:
    print(f"[{idx:4d}]  {im_name}  →  {c_name}")

In [ ]:
# Hand-pick indices from the sample you liked
chosen_indices = [1450,59,1788,594,757,111,1873,898,449]  # whichever you want after visual inspection
chosen_pairs = [all_pairs[i] for i in chosen_indices]

# Write to a new pairs file
with open("/kaggle/working/VITON-HD/test_pairs_model2.txt", "w") as f:
    for im_name, c_name in chosen_pairs:
        f.write(f"{im_name} {c_name}\n")

print(f"Will run inference on {len(chosen_pairs)} images")

In [ ]:
with open("/kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py", "r") as f:
    content = f.read()

# Add argument
content = content.replace(
    '    args = parser.parse_args()',
    '    parser.add_argument("--pairs_txt", type=str, default=None, help="Custom pairs file. Defaults to test_pairs_unpaired.txt")\n    args = parser.parse_args()'
)

# Use it in VITONHDTestDataset.load_data()
content = content.replace(
    "assert os.path.exists(pair_txt:=os.path.join(self.args.data_root_path, 'test_pairs_unpaired.txt'))",
    "pair_txt = self.args.pairs_txt if self.args.pairs_txt else os.path.join(self.args.data_root_path, 'test_pairs_unpaired.txt')\n        assert os.path.exists(pair_txt)"
)

with open("/kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py", "w") as f:
    f.write(content)

print("Verify:")
!grep -n "pairs_txt" /kaggle/working/CatVTON-GenAI-Virtual-Try-On/inference.py

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python inference.py \
  --dataset_name vitonhd \
  --data_root_path /kaggle/working/VITON-HD \
  --output_dir /kaggle/working/output_custom \
  --pairs_txt /kaggle/working/VITON-HD/test_pairs_model2.txt \
  --dataloader_num_workers 2 \
  --batch_size 4 \
  --seed 555 \
  --mixed_precision fp16 \
  --allow_tf32 \
  --repaint

In [ ]:
import os, random
from PIL import Image
import matplotlib.pyplot as plt

image_dir = "/kaggle/working/VITON-HD/test/image"
cloth_dir = "/kaggle/working/VITON-HD/test/cloth"
result_dir = "/kaggle/working/output_custom/vitonhd-512/unpaired"


with open("/kaggle/working/VITON-HD/test_pairs_model2.txt") as f:
    all_pairs = [line.strip().split() for line in f.readlines()]

# Only keep pairs where result exists
available = [
    (im, cl) for im, cl in all_pairs
    if os.path.exists(f"{result_dir}/{im}")
]

sample = random.sample(available, min(10, len(available)))


fig, axes = plt.subplots(len(sample), 3, figsize=(9, len(sample)*3))
for i, (im_name, cl_name) in enumerate(sample):
    axes[i,0].imshow(Image.open(f"{image_dir}/{im_name}"))
    axes[i,0].set_title("Person", fontsize=8)
    axes[i,0].axis("off")

    axes[i,1].imshow(Image.open(f"{cloth_dir}/{cl_name}"))
    axes[i,1].set_title("Cloth", fontsize=8)
    axes[i,1].axis("off")

    axes[i,2].imshow(Image.open(f"{result_dir}/{im_name}"))
    axes[i,2].set_title("Result", fontsize=8)
    axes[i,2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/tryon_comparison.png", dpi=150)
plt.show()

## On the wild images

In [ ]:
# import os
# densepose_dir = "/kaggle/working/CatVTON-GenAI-Virtual-Try-On/densepose"
# os.makedirs(densepose_dir, exist_ok=True)

# # Download config file
# !wget -q https://raw.githubusercontent.com/facebookresearch/detectron2/main/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml \
#     -O {densepose_dir}/densepose_rcnn_R_50_FPN_s1x.yaml

# # Download model weights
# !wget -q https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl \
#     -O {densepose_dir}/model_final_162be9.pkl

# # Download SCHP weights
# schp_dir = "/kaggle/working/CatVTON-GenAI-Virtual-Try-On/resource/schp"
# os.makedirs(schp_dir, exist_ok=True)

# !wget -q https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/humanparsing/exp-schp-201908301523-atr.pth \
#     -O {schp_dir}/exp-schp-201908301523-atr.pth

# !wget -q https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/humanparsing/exp-schp-201908261155-lip.pth \
#     -O {schp_dir}/exp-schp-201908261155-lip.pth

# print("Downloads done. Verifying...")
# print(os.listdir(densepose_dir))
# print(os.listdir(schp_dir))

#-----------------------------------------

import os
densepose_dir = "/kaggle/working/CatVTON-GenAI-Virtual-Try-On/densepose"
os.makedirs(densepose_dir, exist_ok=True)

# Download config file
!wget -q "https://raw.githubusercontent.com/facebookresearch/detectron2/main/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml" \
    -O "{densepose_dir}/densepose_rcnn_R_50_FPN_s1x.yaml"

# Download model weights (~243MB)
!wget -q "https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl" \
    -O "{densepose_dir}/densepose_rcnn_R_50_FPN_s1x.pkl"

# Download SCHP weights
schp_dir = "/kaggle/working/CatVTON-GenAI-Virtual-Try-On/resource/schp"
os.makedirs(schp_dir, exist_ok=True)

!wget -q "https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/humanparsing/exp-schp-201908301523-atr.pth" \
    -O "{schp_dir}/exp-schp-201908301523-atr.pth"

!wget -q "https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/humanparsing/exp-schp-201908261155-lip.pth" \
    -O "{schp_dir}/exp-schp-201908261155-lip.pth"

print("Done! Checking files:")
print(os.listdir(densepose_dir))
print(os.listdir(schp_dir))

In [ ]:
# Peek inside the yaml to see what base configs it needs
!head -5 "{densepose_dir}/densepose_rcnn_R_50_FPN_s1x.yaml"

In [ ]:
densepose_dir = "/kaggle/working/CatVTON-GenAI-Virtual-Try-On/densepose"

# Base config that the main yaml inherits from
!wget -q "https://raw.githubusercontent.com/facebookresearch/detectron2/main/projects/DensePose/configs/Base-DensePose-RCNN-FPN.yaml" \
    -O "{densepose_dir}/Base-DensePose-RCNN-FPN.yaml"

# Check what Base-DensePose-RCNN-FPN.yaml inherits from
!head -3 "{densepose_dir}/Base-DensePose-RCNN-FPN.yaml"

In [ ]:
# These are the typical parent configs in the detectron2 hierarchy
base_cfg_dir = "/kaggle/working/CatVTON-GenAI-Virtual-Try-On/detectron2/configs"
os.makedirs(base_cfg_dir, exist_ok=True)

!wget -q "https://raw.githubusercontent.com/facebookresearch/detectron2/main/configs/Base-RCNN-FPN.yaml" \
    -O "{base_cfg_dir}/Base-RCNN-FPN.yaml"

!wget -q "https://raw.githubusercontent.com/facebookresearch/detectron2/main/configs/Base-RCNN-C4.yaml" \
    -O "{base_cfg_dir}/Base-RCNN-C4.yaml"

print("Files in densepose dir:", os.listdir(densepose_dir))
print("Files in base cfg dir:", os.listdir(base_cfg_dir))

In [ ]:
import os
os.makedirs("/kaggle/working/wild/person", exist_ok=True)
os.makedirs("/kaggle/working/wild/cloth", exist_ok=True)
os.makedirs("/kaggle/working/wild/agnostic-mask", exist_ok=True)

# Upload your person images to /wild/person/
# Upload your cloth images to /wild/cloth/

In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/CatVTON-GenAI-Virtual-Try-On")
os.chdir("/kaggle/working/CatVTON-GenAI-Virtual-Try-On")  # needed for relative config paths

from PIL import Image
from model.cloth_masker import AutoMasker  # correct import, not preprocess_agnostic_mask

os.makedirs("/kaggle/working/wild/agnostic-mask", exist_ok=True)

automasker = AutoMasker(
    densepose_ckpt="./densepose",
    schp_ckpt="./resource/schp",
    device="cuda"
)

person_dir = "/kaggle/working/wild/person"
mask_dir = "/kaggle/working/wild/agnostic-mask"


import requests

urls_person = [
    "https://img.tatacliq.com/images/i11/1348Wx2000H/MP000000017544486_1348Wx2000H_202305151030131.jpeg",
    "https://uspoloassn.in/cdn/shop/files/1_673b06e1-542a-466d-87f0-e60a041565d8_500x.jpg?v=1771499542",
    "https://fashionexport.in/assets/admin/uploads/products/1743602923.JPG"
]

urls_cloth = [
    "https://littleboxindia.com/cdn/shop/files/Sheer_Mesh_Long_Sleeve_Ruched_Wrap_Top_In_Coco_Brown_1024x1024.webp?v=1769845416",
    "https://littleboxindia.com/cdn/shop/files/Casual_Cowl_Neck_Ruched_Side_Tie_Full_Sleeve_Top_in_Yellow.webp?v=1769838823",
    "https://littleboxindia.com/cdn/shop/files/Square_Neck_Floral_Tie_Strap_Sleeveless_Crop_Top_in_Pink_1024x1024.jpg?v=1769843547"
]

for i, url in enumerate(urls_person):
    path = f"/kaggle/working/wild/person/person_{i}.jpg"
    
    response = requests.get(url)
    with open(path, "wb") as f:
        f.write(response.content)

for i, url in enumerate(urls_cloth):
    path = f"/kaggle/working/wild/cloth/cloth_{i}.jpg"
    
    response = requests.get(url)
    with open(path, "wb") as f:
        f.write(response.content)


for fname in os.listdir(person_dir):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    person_image = Image.open(f"{person_dir}/{fname}").convert("RGB")
    person_image = person_image.resize((384, 512))
    
    result = automasker(person_image, "upper")  # "upper" | "lower" | "overall"
    mask = result["mask"]
    
    mask_name = fname.rsplit('.', 1)[0] + '_mask.png'  # cleaner extension replacement
    mask.save(f"{mask_dir}/{mask_name}")
    print(f"Generated mask: {mask_name}")

print(f"\nDone! {len(os.listdir(mask_dir))} masks generated.")

In [ ]:
from PIL import Image, ImageOps
import os

def resize_and_pad(image, target_w=384, target_h=512):
    # Resize keeping aspect ratio, then pad to exact size
    image.thumbnail((target_w, target_h), Image.LANCZOS)
    padded = ImageOps.pad(image, (target_w, target_h), color=(128, 128, 128))
    return padded

person_dir = "/kaggle/working/wild/person"
mask_dir = "/kaggle/working/wild/agnostic-mask"

for fname in os.listdir(person_dir):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    person_image = Image.open(f"{person_dir}/{fname}").convert("RGB")
    print(f"Original size: {person_image.size}")
    
    person_resized = resize_and_pad(person_image)
    print(f"Resized to: {person_resized.size}")
    
    result = automasker(person_resized, "upper")
    mask = result["mask"]
    
    # Visualize immediately
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(10, 4))
    axes[0].imshow(person_resized); axes[0].set_title("Person (resized)")
    axes[1].imshow(mask, cmap='gray'); axes[1].set_title("Mask")
    axes[2].imshow(person_resized); axes[2].imshow(mask, cmap='Reds', alpha=0.5); axes[2].set_title("Overlay")
    plt.show()

    
    
    mask_name = fname.rsplit('.', 1)[0] + '_mask.png'
    mask.save(f"{mask_dir}/{mask_name}")
    print(f"Saved: {mask_name}")

In [ ]:
# Create pairs file
with open("/kaggle/working/wild/wild_pairs.txt", "w") as f:
    for person in sorted(os.listdir("/kaggle/working/wild/person")):
        if not person.lower().endswith(('.jpg','.jpeg','.png')): continue
        for cloth in sorted(os.listdir("/kaggle/working/wild/cloth")):
            if not cloth.lower().endswith(('.jpg','.jpeg','.png')): continue
            f.write(f"{person} {cloth}\n")

!cat /kaggle/working/wild/wild_pairs.txt  # verify

In [ ]:
import os

os.makedirs("/kaggle/working/wild/test", exist_ok=True)

# Symlink your folders to where the script expects them
links = {
    "/kaggle/working/wild/test/image":         "/kaggle/working/wild/person",
    "/kaggle/working/wild/test/cloth":          "/kaggle/working/wild/cloth",
    "/kaggle/working/wild/test/agnostic-mask":  "/kaggle/working/wild/agnostic-mask",
}

for dst, src in links.items():
    if os.path.islink(dst): os.remove(dst)
    os.symlink(src, dst)
    print(f"{dst} → {src}")

# Verify
for dst in links:
    print(f"{dst}: {os.listdir(dst)[:3]}")

In [ ]:
os.chdir("/kaggle/working/CatVTON-GenAI-Virtual-Try-On")

!CUDA_VISIBLE_DEVICES=0 python inference.py \
  --dataset_name vitonhd \
  --data_root_path /kaggle/working/wild \
  --output_dir /kaggle/working/output_wild \
  --pairs_txt /kaggle/working/wild/wild_pairs.txt \
  --dataloader_num_workers 0 \
  --batch_size 1 \
  --seed 555 \
  --mixed_precision fp16 \
  --allow_tf32 \
  --repaint

In [ ]:
from IPython.display import display as ipy_display
from PIL import Image

person = Image.open("/kaggle/working/wild/person/person_0.jpg")
mask = Image.open("/kaggle/working/wild/agnostic-mask/person_0_mask.png")

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow(person); axes[0].set_title("Person"); axes[0].axis("off")
axes[1].imshow(mask, cmap='gray'); axes[1].set_title("Mask"); axes[1].axis("off")
axes[2].imshow(person)
axes[2].imshow(np.array(mask), cmap='Reds', alpha=0.5)
axes[2].set_title("Overlay"); axes[2].axis("off")
plt.tight_layout()
plt.show()

print("HI")

In [ ]:
ipy_display(person)
ipy_display(mask)